# SQLAlchemy and Relational Thinking
Database libraries get much easier to understand when you keep two pictures in mind at once: Python objects in your process and relational rows in a database engine. SQLAlchemy is valuable because it helps you move between those worlds without forgetting they are still different worlds.

SQLAlchemy can be used in both lower-level SQL expression style and higher-level ORM style. The common deep skill is learning to think relationally about tables, keys, joins, transactions, and persistence.

In [1]:
from sqlalchemy import create_engine, Column, Integer, String, select
from sqlalchemy.orm import declarative_base, Session

Base = declarative_base()

class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    name = Column(String)
    city = Column(String)

engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

with Session(engine) as session:
    session.add_all([User(name='Anika', city='Hyd'), User(name='Rahul', city='Blr')])
    session.commit()
    rows = session.execute(select(User)).scalars().all()
    for row in rows:
        print(row.id, row.name, row.city)

1 Anika Hyd
2 Rahul Blr


The ORM style is often easier to read for Python developers, but the database still thinks in terms of tables, rows, constraints, and query plans. Good SQLAlchemy users remember that the abstraction is helpful, not magical.

In [2]:
with Session(engine) as session:
    user = session.execute(select(User).where(User.city == 'Hyd')).scalar_one()
    user.city = 'Pune'
    session.commit()

with Session(engine) as session:
    rows = session.execute(select(User)).scalars().all()
    print([(row.name, row.city) for row in rows])

[('Anika', 'Pune'), ('Rahul', 'Blr')]


A session tracks object state and coordinates database communication. In a rough mental model, it acts like a unit-of-work manager that decides what inserts, updates, or deletes need to be flushed to the database.

In [3]:
with Session(engine) as session:
    user = session.execute(select(User).where(User.name == 'Rahul')).scalar_one()
    session.delete(user)
    session.commit()

with Session(engine) as session:
    print(session.execute(select(User)).scalars().all())

Transactions matter because persistence should not become half-finished silently. The deeper question is not just how to write ORM code, but how to preserve consistency when multiple changes belong together.

In [4]:
relational_questions = [
    'What is the primary key?',
    'What should be unique?',
    'What rows should survive deletion of related records?',
    'What shape should joins return?',
    'What belongs in Python validation versus database constraints?'
]
for question in relational_questions:
    print('-', question)

- What is the primary key?
- What should be unique?
- What rows should survive deletion of related records?
- What shape should joins return?
- What belongs in Python validation versus database constraints?


For interviews, it helps to explain SQLAlchemy as a bridge between Python objects and SQL databases, while still emphasizing sessions, transactions, schema design, and the importance of understanding the generated SQL behavior.

## Final Takeaway
SQLAlchemy is most useful when you keep the abstraction honest. Python objects are convenient handles, but the underlying truth is relational data, transactional integrity, and query behavior inside the database engine.